In [19]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [20]:
df = pd.read_csv("combined/train_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1006, 44)


,rssi_RX1,rssi_RX2,rssi_RX3,rssi_diff_12,rssi_diff_13,rssi_diff_23,RX1_mean,RX1_std,RX1_energy,RX1_max,...,RX1_RX2_diff_std,RX1_RX3_diff_mean,RX1_RX3_diff_std,RX2_RX3_diff_mean,RX2_RX3_diff_std,RX1_RX2_corr,RX1_RX3_corr,RX2_RX3_corr,scenario_id,person_count
0,-40.566667,-47.533333,-61.700000,6.966667,21.133333,14.166667,19.274798,19.504202,1443709.0,148.165448,...,7.128848,3.101467,11.229125,2.584204,11.214379,0.932957,0.837270,0.837152,No_person,0
1,-40.666667,-47.333333,-61.500000,6.666667,20.833333,14.166667,19.305673,19.541671,1448805.0,148.165448,...,7.770728,2.989139,11.221650,2.535102,10.799333,0.920477,0.836507,0.847735,No_person,0
2,-40.533333,-47.366667,-61.500000,6.833333,20.966667,14.133333,19.008895,19.619012,1432788.0,148.165448,...,7.122351,2.455238,11.464241,2.018441,10.857040,0.933466,0.830205,0.846229,No_person,0
3,-40.166667,-47.333333,-61.433333,7.166667,21.266667,14.100000,18.583656,19.584029,1399462.0,148.165448,...,7.540916,1.733639,11.761718,1.786424,10.603654,0.925300,0.821012,0.853444,No_person,0
4,-39.733333,-47.166667,-61.433333,7.433333,21.700000,14.266667,17.886133,19.458377,1341201.0,148.165448,...,8.097918,1.113196,11.527767,1.999029,10.617011,0.913161,0.827214,0.853084,No_person,0


In [21]:
y = df["person_count"]
X = df.drop(columns=["scenario_id", "person_count"])
groups = df["scenario_id"]

print("Total samples:", len(X))
print("Unique scenarios:", groups.nunique())

print("\nOverall class distribution:")
print(y.value_counts().sort_index())

Total samples: 1006
Unique scenarios: 3

Overall class distribution:
person_count
0    467
1    529
2     10
Name: count, dtype: int64


In [22]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,
    random_state=42
)

print("Train classes:", sorted(y_train.unique()))
print("Validation classes:", sorted(y_val.unique()))

print("\nTrain distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation distribution:")
print(y_val.value_counts().sort_index())

Train classes: [0, 1, 2]
Validation classes: [0, 1, 2]

Train distribution:
person_count
0    420
1    476
2      9
Name: count, dtype: int64

Validation distribution:
person_count
0    47
1    53
2     1
Name: count, dtype: int64


In [23]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

print("Scaling applied.")

Scaling applied.


In [24]:
rf_model = RandomForestClassifier(
    n_estimators=700,
    max_depth=20,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Training accuracy
rf_train_preds = rf_model.predict(X_train)
rf_train_acc = accuracy_score(y_train, rf_train_preds)

print("✅ Random Forest trained")
print(f"Training Accuracy: {rf_train_acc:.4f}")

✅ Random Forest trained
Training Accuracy: 0.9967


In [25]:
rf_preds = rf_model.predict(X_val)

val_acc = accuracy_score(y_val, rf_preds)

print("=== Random Forest Results ===\n")

print(f"Training Accuracy: {rf_train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_preds))

print("\nClassification Report:")
print(classification_report(y_val, rf_preds, zero_division=0))

mae = np.mean(np.abs(y_val.values - rf_preds))
print("\nMean Absolute Error:", mae)

tol_acc = np.mean(np.abs(y_val.values - rf_preds) <= 1)
print("Accuracy within ±1 person:", tol_acc)

=== Random Forest Results ===

Training Accuracy: 0.9967
Validation Accuracy: 0.9406

Confusion Matrix:
[[44  3  0]
 [ 3 50  0]
 [ 0  0  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        47
           1       0.94      0.94      0.94        53
           2       1.00      1.00      1.00         1

    accuracy                           0.94       101
   macro avg       0.96      0.96      0.96       101
weighted avg       0.94      0.94      0.94       101


Mean Absolute Error: 0.0594059405940594
Accuracy within ±1 person: 1.0


In [26]:
import os

# Create models directory if it doesn't exist
models_dir = "../grad_dashboard/backend/models"
os.makedirs(models_dir, exist_ok=True)

# Save model, scaler, and features to the backend models directory
joblib.dump(rf_model, os.path.join(models_dir, "rf_person_count_model.pkl"))
joblib.dump(scaler, os.path.join(models_dir, "scaler.pkl"))
joblib.dump(X.columns.tolist(), os.path.join(models_dir, "feature_columns.pkl"))

print(f"✅ Model, scaler, and features saved to {models_dir}")

✅ Model, scaler, and features saved to ../grad_dashboard/backend/models


In [27]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Load data again
df = pd.read_csv("combined/train_dataset.csv")

# Prepare data
y = df["person_count"]
X = df.drop(columns=["scenario_id", "person_count"])

# Split (same as your training)
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

# 🔥 Shuffle labels (THIS is the key test)
y_train_shuffled = y_train.sample(frac=1, random_state=42).reset_index(drop=True)

# Train model on WRONG labels
rf_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model.fit(X_train, y_train_shuffled)

# Evaluate
preds = rf_model.predict(X_val)
acc = accuracy_score(y_val, preds)

print("Accuracy after label shuffle:", acc)

Accuracy after label shuffle: 0.3712871287128713
